# BERT 虚假新闻检测模型

本 Notebook 将原 ResNet/CNN 文本分类模型替换为普通 BERT 二分类模型。数据集路径保持不变：

- 训练集：`E:/Fakenews Detection/Dataest_2.0/Fakenews_train.xlsx`
- 测试集：`E:/Fakenews Detection/Dataest_2.0/Fakenews_test.xlsx`

数据表需包含：

- `text`：新闻文本
- `label`：新闻标签，通常为 0/1

In [ ]:
# 如未安装依赖，请先取消注释运行
# !pip install pandas openpyxl scikit-learn matplotlib torch transformers

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification, AdamW, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt

# 固定随机种子，保证实验尽量可复现
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('当前运行设备：', DEVICE)

In [ ]:
# 数据集路径保持不变
TRAIN_PATH = 'E:/Fakenews Detection/Dataest_2.0/Fakenews_train.xlsx'
TEST_PATH = 'E:/Fakenews Detection/Dataest_2.0/Fakenews_test.xlsx'

train_df = pd.read_excel(TRAIN_PATH, engine='openpyxl')
test_df = pd.read_excel(TEST_PATH, engine='openpyxl')

print('训练集大小：', train_df.shape)
print('测试集大小：', test_df.shape)
print('训练集字段：', train_df.columns.tolist())

train_df.head()

In [ ]:
# 数据清洗：保证 text 和 label 字段可用
required_cols = ['text', 'label']
for col in required_cols:
    if col not in train_df.columns or col not in test_df.columns:
        raise ValueError(f'数据集中缺少必要字段：{col}')

train_df = train_df[['text', 'label']].copy()
test_df = test_df[['text', 'label']].copy()

train_df['text'] = train_df['text'].fillna('').astype(str)
test_df['text'] = test_df['text'].fillna('').astype(str)

train_df['label'] = train_df['label'].fillna(0).astype(int)
test_df['label'] = test_df['label'].fillna(0).astype(int)

print('训练集标签分布：')
print(train_df['label'].value_counts())
print('
测试集标签分布：')
print(test_df['label'].value_counts())

In [ ]:
# BERT 配置
MODEL_NAME = 'bert-base-uncased'
MAX_LEN = 128
BATCH_SIZE = 8
EPOCHS = 3
LEARNING_RATE = 2e-5

# 加载 BERT tokenizer
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
class FakeNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

train_dataset = FakeNewsDataset(
    train_df['text'].values,
    train_df['label'].values,
    tokenizer,
    MAX_LEN
)

test_dataset = FakeNewsDataset(
    test_df['text'].values,
    test_df['label'].values,
    tokenizer,
    MAX_LEN
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print('训练批次数：', len(train_loader))
print('测试批次数：', len(test_loader))

In [ ]:
# 构建普通 BERT 二分类模型
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)
model = model.to(DEVICE)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

print(model)

In [ ]:
def train_one_epoch(model, data_loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []

    for batch in data_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1).detach().cpu().numpy()
        labels_np = labels.detach().cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels_np)

    avg_loss = total_loss / len(data_loader)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc


def evaluate(model, data_loader, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            logits = outputs.logits
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1)

            total_loss += loss.item()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())

    avg_loss = total_loss / len(data_loader)
    acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)

    return {
        'loss': avg_loss,
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'labels': all_labels,
        'preds': all_preds,
        'probs': all_probs
    }

In [ ]:
# 开始训练
os.makedirs('bert_results', exist_ok=True)

history = {
    'train_loss': [],
    'train_acc': [],
    'test_loss': [],
    'test_acc': [],
    'test_precision': [],
    'test_recall': [],
    'test_f1': []
}

best_f1 = 0.0
best_model_path = 'bert_results/best_bert_fake_news_model'

for epoch in range(EPOCHS):
    print(f'
Epoch {epoch + 1}/{EPOCHS}')
    print('-' * 50)

    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, scheduler, DEVICE)
    test_metrics = evaluate(model, test_loader, DEVICE)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(test_metrics['loss'])
    history['test_acc'].append(test_metrics['accuracy'])
    history['test_precision'].append(test_metrics['precision'])
    history['test_recall'].append(test_metrics['recall'])
    history['test_f1'].append(test_metrics['f1'])

    print(f'训练损失: {train_loss:.4f} | 训练准确率: {train_acc:.4f}')
    print(
        f"测试损失: {test_metrics['loss']:.4f} | "
        f"Accuracy: {test_metrics['accuracy']:.4f} | "
        f"Precision: {test_metrics['precision']:.4f} | "
        f"Recall: {test_metrics['recall']:.4f} | "
        f"F1: {test_metrics['f1']:.4f}"
    )

    if test_metrics['f1'] > best_f1:
        best_f1 = test_metrics['f1']
        model.save_pretrained(best_model_path)
        tokenizer.save_pretrained(best_model_path)
        print(f'已保存当前最优模型，F1 = {best_f1:.4f}')

In [ ]:
# 最终测试集评估
final_metrics = evaluate(model, test_loader, DEVICE)

print('最终测试集结果：')
print(f"Accuracy : {final_metrics['accuracy']:.4f}")
print(f"Precision: {final_metrics['precision']:.4f}")
print(f"Recall   : {final_metrics['recall']:.4f}")
print(f"F1-score : {final_metrics['f1']:.4f}")

print('分类报告：')
print(classification_report(final_metrics['labels'], final_metrics['preds'], digits=4, zero_division=0))

print('混淆矩阵：')
print(confusion_matrix(final_metrics['labels'], final_metrics['preds']))

In [ ]:
# 保存预测结果
result_df = test_df.copy()
result_df['pred_label'] = final_metrics['preds']
result_df['fake_probability'] = final_metrics['probs']
result_df['predict_result'] = result_df['pred_label'].map({0: '真实新闻', 1: '虚假新闻'})

save_path = 'bert_results/bert_test_predictions.xlsx'
result_df.to_excel(save_path, index=False)
print('预测结果已保存至：', save_path)

result_df.head()

In [ ]:
# 绘制训练过程曲线
plt.figure(figsize=(8, 5))
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['test_loss'], label='Test Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('BERT Training Loss')
plt.legend()
plt.tight_layout()
plt.savefig('bert_results/bert_loss_curve.png', dpi=300)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history['train_acc'], label='Train Accuracy')
plt.plot(history['test_acc'], label='Test Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('BERT Accuracy')
plt.legend()
plt.tight_layout()
plt.savefig('bert_results/bert_accuracy_curve.png', dpi=300)
plt.show()

In [ ]:
# 单条新闻测试函数

def predict_single_news(text):
    model.eval()
    encoding = tokenizer.encode_plus(
        str(text),
        add_special_tokens=True,
        max_length=MAX_LEN,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(DEVICE)
    attention_mask = encoding['attention_mask'].to(DEVICE)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]
        pred = int(np.argmax(probs))

    label_name = '虚假新闻' if pred == 1 else '真实新闻'
    confidence = float(probs[pred])

    return {
        '预测标签': pred,
        '预测结果': label_name,
        '置信度': confidence,
        '真实概率': float(probs[0]),
        '虚假概率': float(probs[1])
    }

# 示例：可替换为任意新闻文本
sample_text = 'Breaking news: scientists discovered a new method for improving public health.'
predict_single_news(sample_text)